# Step 3: Feature Engineering

Input: `data/train_cleaned.parquet`, `data/test_cleaned.parquet`  
Output: `data/features_train.parquet`, `data/features_test.parquet`

Features built:
1. Geo-aware holiday flags (match regional/local holidays to store's state/city)
2. Date features (day_of_week, month, year, is_weekend, etc.)
3. Promotion features
4. Oil price rolling features
5. Lag features — grouped by (store_nbr, family)
6. Rolling statistics — grouped by (store_nbr, family)
7. Categorical encoding
8. Drop unused columns & save

In [10]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

DATA_DIR = 'data/'

## 1. Load Cleaned Data

In [11]:
train = pd.read_parquet(DATA_DIR + 'train_cleaned.parquet')
test  = pd.read_parquet(DATA_DIR + 'test_cleaned.parquet')

# Load raw files needed for geo-aware holiday correction
holidays = pd.read_csv(DATA_DIR + 'holidays_events.csv', parse_dates=['date'])
stores   = pd.read_csv(DATA_DIR + 'stores.csv')

print(f'train: {train.shape}  |  test: {test.shape}')
print('Columns:', train.columns.tolist())

train: (3000888, 21)  |  test: (28512, 20)
Columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'store_type', 'cluster', 'oil_price', 'national_holiday_type', 'national_holiday_name', 'is_national_holiday', 'regional_holiday_type', 'reg_holiday_name', 'is_regional_holiday', 'local_holiday_type', 'loc_holiday_name', 'is_local_holiday', 'transactions']


## 2. Fix Geo-Aware Holiday Flags

Step 2 merged holidays on `date` only — every store was flagged for every regional/local holiday regardless of location.  
Here we rebuild correct flags:
- **Regional**: holiday `locale_name` must match the store's `state`
- **Local**: holiday `locale_name` must match the store's `city`

In [12]:
# Drop ALL holiday columns from Step 2 — we rebuild them geo-aware below
drop_cols = [
    'national_holiday_type', 'national_holiday_name', 'is_national_holiday',
    'regional_holiday_type', 'reg_holiday_name',       'is_regional_holiday',
    'local_holiday_type',    'loc_holiday_name',        'is_local_holiday'
]
train = train.drop(columns=[c for c in drop_cols if c in train.columns])
test  = test.drop(columns=[c for c in drop_cols if c in test.columns])

# Effective holidays only
hol = holidays[holidays['transferred'] == False].copy()
hol['type'] = hol['type'].replace('Transfer', 'Holiday')

# National: date-level flag (already correct in step 2, but rebuild cleanly)
nat = hol[hol['locale'] == 'National'][['date']].drop_duplicates().assign(is_national_holiday=True)

# Regional: (date, state) level
reg = (
    hol[hol['locale'] == 'Regional'][['date', 'locale_name']]
    .drop_duplicates()
    .rename(columns={'locale_name': 'state'})
    .assign(is_regional_holiday=True)
)

# Local: (date, city) level
loc = (
    hol[hol['locale'] == 'Local'][['date', 'locale_name']]
    .drop_duplicates()
    .rename(columns={'locale_name': 'city'})
    .assign(is_local_holiday=True)
)

def add_holidays(df):
    df = df.merge(nat, on='date', how='left')
    df['is_national_holiday'] = df['is_national_holiday'].fillna(False)

    df = df.merge(reg, on=['date', 'state'], how='left')
    df['is_regional_holiday'] = df['is_regional_holiday'].fillna(False)

    df = df.merge(loc, on=['date', 'city'], how='left')
    df['is_local_holiday'] = df['is_local_holiday'].fillna(False)

    # Combined flag: any holiday affecting this store today
    df['is_holiday'] = (
        df['is_national_holiday'] | df['is_regional_holiday'] | df['is_local_holiday']
    )
    return df

train = add_holidays(train)
test  = add_holidays(test)

print('Holiday flags (train):')
print(train[['is_national_holiday','is_regional_holiday','is_local_holiday','is_holiday']].sum())

Holiday flags (train):
is_national_holiday    242352
is_regional_holiday      1023
is_local_holiday        11880
is_holiday             254760
dtype: int64


## 3. Combine Train + Test

Lag and rolling features require looking back from test dates into train data.  
We concatenate, compute all features on the full timeline, then split back.

In [13]:
train['is_train'] = True
test['is_train']  = False

# test has no sales column — add it as NaN so concat works cleanly
if 'sales' not in test.columns:
    test['sales'] = np.nan

full = pd.concat([train, test], sort=False)
full = full.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

print(f'Combined shape: {full.shape}')

Combined shape: (3029400, 17)


## 4. Date Features

In [14]:
full['day_of_week']  = full['date'].dt.dayofweek          # 0=Mon, 6=Sun
full['day_of_month'] = full['date'].dt.day
full['day_of_year']  = full['date'].dt.dayofyear
full['week_of_year'] = full['date'].dt.isocalendar().week.astype(int)
full['month']        = full['date'].dt.month
full['quarter']      = full['date'].dt.quarter
full['year']         = full['date'].dt.year
full['is_weekend']   = (full['day_of_week'] >= 5).astype(int)
full['is_month_start'] = full['date'].dt.is_month_start.astype(int)
full['is_month_end']   = full['date'].dt.is_month_end.astype(int)

print('Date features added:', ['day_of_week','day_of_month','day_of_year',
      'week_of_year','month','quarter','year','is_weekend',
      'is_month_start','is_month_end'])

Date features added: ['day_of_week', 'day_of_month', 'day_of_year', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend', 'is_month_start', 'is_month_end']


## 5. Promotion Features

In [15]:
# Binary flag + keep the raw count
full['is_on_promo'] = (full['onpromotion'] > 0).astype(int)

# Log-scale of promotion count (handles skewness; onpromotion max = 741)
full['log_onpromotion'] = np.log1p(full['onpromotion'])

print('Promotion features: is_on_promo, log_onpromotion')

Promotion features: is_on_promo, log_onpromotion


## 6. Oil Price Rolling Features

In [16]:
# oil_price is already at date level — compute rolling stats on the daily series
oil_daily = full[['date', 'oil_price']].drop_duplicates('date').sort_values('date').copy()
oil_daily['oil_ma7']  = oil_daily['oil_price'].rolling(7,  min_periods=1).mean()
oil_daily['oil_ma28'] = oil_daily['oil_price'].rolling(28, min_periods=1).mean()

full = full.merge(oil_daily[['date','oil_ma7','oil_ma28']], on='date', how='left')

print('Oil features: oil_price (existing), oil_ma7, oil_ma28')

Oil features: oil_price (existing), oil_ma7, oil_ma28


## 7. Lag Features

Grouped by `(store_nbr, family)` — each group is an independent time series.

| Feature | Meaning |
|---------|--------|
| `lag_7`  | Sales same day last week |
| `lag_14` | Sales same day 2 weeks ago |
| `lag_28` | Sales same day 4 weeks ago |
| `lag_35` | Sales same day 5 weeks ago |

> Note: for the test period (Aug 16–31), `lag_7` is NaN for Aug 23–31 and `lag_14` is NaN for Aug 30–31 because those lookback dates fall inside the test window. `lag_28` and `lag_35` are valid for all test rows. LightGBM handles NaN natively.

In [17]:
grp = full.groupby(['store_nbr', 'family'])['sales']

for lag in [7, 14, 28, 35]:
    full[f'lag_{lag}'] = grp.shift(lag)

lag_cols = ['lag_7', 'lag_14', 'lag_28', 'lag_35']
print('Lag features added:', lag_cols)
print('\nNaN counts in lag features (train only):')
train_mask = full['is_train']
print(full.loc[train_mask, lag_cols].isnull().sum())

Lag features added: ['lag_7', 'lag_14', 'lag_28', 'lag_35']

NaN counts in lag features (train only):
lag_7     12474
lag_14    24948
lag_28    49896
lag_35    62370
dtype: int64


## 8. Rolling Statistics

All rolling windows use `shift(1)` before rolling to avoid data leakage  
(we must not include the current day's sales when predicting that day).

In [18]:
def rolling_feature(series, window, func='mean'):
    shifted = series.shift(1)   # exclude current day
    if func == 'mean':
        return shifted.rolling(window, min_periods=1).mean()
    elif func == 'std':
        return shifted.rolling(window, min_periods=1).std()
    elif func == 'max':
        return shifted.rolling(window, min_periods=1).max()

grp = full.groupby(['store_nbr', 'family'])['sales']

full['rolling_mean_7']  = grp.transform(lambda x: rolling_feature(x, 7,  'mean'))
full['rolling_mean_14'] = grp.transform(lambda x: rolling_feature(x, 14, 'mean'))
full['rolling_mean_28'] = grp.transform(lambda x: rolling_feature(x, 28, 'mean'))
full['rolling_std_7']   = grp.transform(lambda x: rolling_feature(x, 7,  'std'))
full['rolling_max_28']  = grp.transform(lambda x: rolling_feature(x, 28, 'max'))

rolling_cols = ['rolling_mean_7','rolling_mean_14','rolling_mean_28','rolling_std_7','rolling_max_28']
print('Rolling features added:', rolling_cols)

Rolling features added: ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_max_28']


## 9. Encode Categorical Features

In [19]:
# Label-encode: family, store_type, city, state
# pd.factorize assigns a consistent integer to each unique string value
for col in ['family', 'store_type', 'city', 'state']:
    codes, uniques = pd.factorize(full[col])
    full[f'{col}_enc'] = codes
    print(f'{col}_enc: {len(uniques)} unique values')

# Holiday bool flags → int (LightGBM works with both, but int is cleaner)
for col in ['is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_holiday']:
    full[col] = full[col].astype(int)

family_enc: 33 unique values
store_type_enc: 5 unique values
city_enc: 22 unique values
state_enc: 16 unique values


## 10. Drop Unused Columns & Split Back

In [20]:
# Columns to drop: raw string categoricals (now encoded), id (not a feature)
DROP = ['city', 'state', 'family', 'store_type',   # replaced by *_enc
        'is_train']                                  # internal split flag

full = full.drop(columns=[c for c in DROP if c in full.columns])

# Split back
# is_train was dropped — use sales.isna() to identify test rows
# (test rows have sales=NaN since we added it artificially)
train_feat = full[full['sales'].notna()].copy()
test_feat  = full[full['sales'].isna()].copy()
test_feat  = test_feat.drop(columns=['sales'])   # clean: test has no sales

print(f'train_feat: {train_feat.shape}')
print(f'test_feat:  {test_feat.shape}')
print('\nFinal columns:')
print(train_feat.columns.tolist())

train_feat: (3000888, 39)
test_feat:  (28512, 38)

Final columns:
['id', 'date', 'store_nbr', 'sales', 'onpromotion', 'cluster', 'oil_price', 'transactions', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_holiday', 'day_of_week', 'day_of_month', 'day_of_year', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend', 'is_month_start', 'is_month_end', 'is_on_promo', 'log_onpromotion', 'oil_ma7', 'oil_ma28', 'lag_7', 'lag_14', 'lag_28', 'lag_35', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_max_28', 'family_enc', 'store_type_enc', 'city_enc', 'state_enc']


## 11. Validation

In [21]:
print('=== train_feat missing values ===')
missing = train_feat.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('  None (all filled)')
else:
    pct = (missing / len(train_feat) * 100).round(2)
    print(pd.DataFrame({'missing': missing, 'pct%': pct}))
    print('  Note: lag NaN at series start is expected (not enough history)')

=== train_feat missing values ===
                 missing  pct%
lag_7              12474  0.42
lag_14             24948  0.83
lag_28             49896  1.66
lag_35             62370  2.08
rolling_mean_7      1782  0.06
rolling_mean_14     1782  0.06
rolling_mean_28     1782  0.06
rolling_std_7       3564  0.12
rolling_max_28      1782  0.06
  Note: lag NaN at series start is expected (not enough history)


In [22]:
print('=== test_feat missing values ===')
missing_t = test_feat.isnull().sum()
missing_t = missing_t[missing_t > 0]
if missing_t.empty:
    print('  None')
else:
    pct_t = (missing_t / len(test_feat) * 100).round(2)
    print(pd.DataFrame({'missing': missing_t, 'pct%': pct_t}))
    print('  Note: lag_7/lag_14 NaN for last ~9/2 days is expected')

=== test_feat missing values ===
                 missing   pct%
lag_7              16038  56.25
lag_14              3564  12.50
rolling_mean_7     16038  56.25
rolling_mean_14     3564  12.50
rolling_std_7      17820  62.50
  Note: lag_7/lag_14 NaN for last ~9/2 days is expected


In [23]:
# Spot-check: no future leakage — rolling features must use shift(1)
# Verify: rolling_mean_7 for 2013-01-08 == mean of sales on 2013-01-01 to 2013-01-07
sample = train_feat[
    (train_feat['store_nbr'] == 1) & (train_feat['family_enc'] == 0)
].sort_values('date').head(15)[['date','sales','lag_7','rolling_mean_7']]
print('Leakage check (store=1, family=0):')
print(sample.to_string(index=False))

Leakage check (store=1, family=0):
      date  sales  lag_7  rolling_mean_7
2013-01-01    0.0    NaN             NaN
2013-01-02    2.0    NaN        0.000000
2013-01-03    3.0    NaN        1.000000
2013-01-04    3.0    NaN        1.666667
2013-01-05    5.0    NaN        2.000000
2013-01-06    2.0    NaN        2.600000
2013-01-07    0.0    NaN        2.500000
2013-01-08    2.0    0.0        2.142857
2013-01-09    2.0    2.0        2.428571
2013-01-10    2.0    3.0        2.428571
2013-01-11    3.0    3.0        2.285714
2013-01-12    2.0    5.0        2.285714
2013-01-13    2.0    2.0        1.857143
2013-01-14    2.0    0.0        1.857143
2013-01-15    1.0    2.0        2.142857


## 12. Save Feature Data

In [24]:
train_feat.to_parquet(DATA_DIR + 'features_train.parquet', index=False)
test_feat.to_parquet(DATA_DIR  + 'features_test.parquet',  index=False)

print('Saved:')
print(f'  data/features_train.parquet  ({train_feat.shape[0]:,} rows x {train_feat.shape[1]} cols)')
print(f'  data/features_test.parquet   ({test_feat.shape[0]:,} rows x {test_feat.shape[1]} cols)')
print(f'\nFeature columns ({train_feat.shape[1]}):')
print(train_feat.columns.tolist())

Saved:
  data/features_train.parquet  (3,000,888 rows x 39 cols)
  data/features_test.parquet   (28,512 rows x 38 cols)

Feature columns (39):
['id', 'date', 'store_nbr', 'sales', 'onpromotion', 'cluster', 'oil_price', 'transactions', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_holiday', 'day_of_week', 'day_of_month', 'day_of_year', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend', 'is_month_start', 'is_month_end', 'is_on_promo', 'log_onpromotion', 'oil_ma7', 'oil_ma28', 'lag_7', 'lag_14', 'lag_28', 'lag_35', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_max_28', 'family_enc', 'store_type_enc', 'city_enc', 'state_enc']
